# Volume 2, Chapter 12: Quaternions and Rotations (Python companion)

Exercises first, solutions at the end. We build a small quaternion class, check the rotation formulas of the chapter, compare quaternions and matrices for composing many rotations (cost and numerical drift), interpolate orientations with slerp, meet gimbal lock, generate the finite rotation groups and their binary covers, and finally use the Euclidean algorithm in the Hurwitz quaternions to write a prime as a sum of four squares.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
import time
from fractions import Fraction
from collections import Counter
import numpy as np
import sympy
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## Exercises

**P12.1 (★) A quaternion class.** Write a class `Q` with addition, multiplication (Proposition 12.2), conjugate, norm and inverse, generic in the type of its coordinates (floats, `Fraction`s or sympy symbols). Check $i^2=j^2=k^2=ijk=-1$, associativity and $N(pq)=N(p)N(q)$ on random examples, and redo Exercise 12.1: the quarter turn about $z$ followed by the quarter turn about $x$.

In [ ]:
# Your code here


**P12.2 (★) The rotation matrix, symbolically.** With sympy, compute $v\mapsto qv\bar q$ on $i,j,k$ for $q=a+bi+cj+dk$, and check the matrix of Exercise 12.3: it is orthogonal with determinant $1$ when $a^2+b^2+c^2+d^2=1$, and its trace is $4a^2-1$. Check that $q$ and $-q$ give the same matrix (Theorem 12.4).

In [ ]:
# Your code here


**P12.3 (★★) Composing a million small rotations.** Compose $10^5$ random small rotations in two ways, as $3\times3$ matrices in single precision and as unit quaternions in single precision, without any correction. Measure how far the matrix product is from orthogonal ($\|R^TR-I\|$) and how far the quaternion is from norm $1$. Then fix each: renormalise the quaternion (one division), re-orthogonalise the matrix (Gram–Schmidt or SVD). Count the multiplications per composition.

In [ ]:
# Your code here


**P12.4 (★★) Interpolating orientations.** Given two orientations $q_0,q_1$, compare three ways of going from one to the other: linear interpolation of the matrices, linear interpolation of Euler angles, and slerp $q_t=q_0(\bar q_0q_1)^t$. Plot the angular speed along each path. Then exhibit *gimbal lock*: with Euler angles (yaw, pitch, roll) at pitch $90^\circ$, show that the map from angles to rotations loses a dimension (rank of its Jacobian).

In [ ]:
# Your code here


**P12.5 (★★) Finite rotation groups and binary groups.** Generate, as sets of unit quaternions, the groups generated by: (a) $\frac12(1+i+j+k)$ and $i$; (b) $\frac1{\sqrt2}(1+i)$ and $\frac12(1+i+j+k)$; (c) $\frac12(1+i+j+k)$ and $\frac12(\varphi+\varphi^{-1}i+j)$, $\varphi$ the golden ratio. Find their orders ($24$, $48$, $120$), the distribution of element orders, the number of elements of order $2$ (Exercise 12.6), and the orders of their images in $SO(3)$ (Theorem 12.5).

In [ ]:
# Your code here


**P12.6 (★★★) Four squares by a Euclidean algorithm.** For a prime $p$, find $m=1+xi+yj$ with $1+x^2+y^2\equiv0\pmod p$ (Volume 1). In the Hurwitz quaternions, run the Euclidean algorithm on $p$ and $m$ (divide with remainder by rounding $ab^{-1}$ to the nearest Hurwitz quaternion) to find a generator $g$ of the left ideal $\mathcal Hp+\mathcal Hm$; then $N(g)=p$ (Exercise 12.7). Multiply by a unit to make the coordinates integers, and read off $p=a^2+b^2+c^2+d^2$. Time it for primes around $10^{30}$, and compare the number of representations of small $n$ with Jacobi's formula $r_4(n)=8\sum_{4\nmid d\mid n}d$.

In [ ]:
# Your code here


---
## Solutions

### P12.1

In [ ]:
class Q:
    __slots__ = ('a', 'b', 'c', 'd')
    def __init__(self, a, b=0, c=0, d=0):
        self.a, self.b, self.c, self.d = a, b, c, d
    def __add__(self, o): return Q(self.a + o.a, self.b + o.b, self.c + o.c, self.d + o.d)
    def __sub__(self, o): return Q(self.a - o.a, self.b - o.b, self.c - o.c, self.d - o.d)
    def __neg__(self): return Q(-self.a, -self.b, -self.c, -self.d)
    def __mul__(self, o):
        if not isinstance(o, Q):
            return Q(self.a * o, self.b * o, self.c * o, self.d * o)
        a, b, c, d = self.a, self.b, self.c, self.d
        e, f, g, h = o.a, o.b, o.c, o.d
        return Q(a*e - b*f - c*g - d*h,
                 a*f + b*e + c*h - d*g,
                 a*g - b*h + c*e + d*f,
                 a*h + b*g - c*f + d*e)
    __rmul__ = lambda self, s: self * s
    def conj(self): return Q(self.a, -self.b, -self.c, -self.d)
    def norm(self): return self.a**2 + self.b**2 + self.c**2 + self.d**2
    def inv(self):
        n = self.norm()
        return Q(self.a / n, -self.b / n, -self.c / n, -self.d / n)
    def tuple(self): return (self.a, self.b, self.c, self.d)
    def __eq__(self, o): return self.tuple() == o.tuple()
    def __hash__(self): return hash(self.tuple())
    def __repr__(self): return f'Q{self.tuple()}'

one, I, J, K = Q(1), Q(0, 1), Q(0, 0, 1), Q(0, 0, 0, 1)
print('i^2, j^2, k^2, ijk =', I*I, J*J, K*K, I*J*K)
print('ij, ji =', I*J, J*I)

rnd = lambda: Q(*[Fraction(random.randint(-9, 9), random.randint(1, 9)) for _ in range(4)])
ok_assoc = ok_norm = True
for _ in range(1000):
    p, q, r = rnd(), rnd(), rnd()
    ok_assoc &= (p*q)*r == p*(q*r)
    ok_norm &= (p*q).norm() == p.norm() * q.norm()
print('associative:', ok_assoc, '   N(pq) = N(p)N(q):', ok_norm)

def rot(q, v):
    """Rotate the 3-vector v by the unit quaternion q: v -> q v q-bar."""
    w = q * Q(0, *v) * q.conj()
    return (w.b, w.c, w.d)

s = 1 / math.sqrt(2)
q1, q2 = Q(s, 0, 0, s), Q(s, s, 0, 0)      # quarter turns about z, then about x
q = q2 * q1
print('q2 q1 =', q.tuple())
theta = 2 * math.degrees(math.acos(q.a))
axis = np.array([q.b, q.c, q.d]) / math.sin(math.radians(theta / 2))
print('angle', round(theta, 6), 'axis', axis * math.sqrt(3))
for e in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]:
    print(e, '->', np.round(rot(q, e), 12))

The class works unchanged with `Fraction` coordinates (exact checks) and with floats. The product $q_2q_1=\frac12(1+i-j+k)$ is a rotation by $120^\circ$ about $\frac1{\sqrt3}(1,-1,1)$, and it sends $e_1\mapsto e_3\mapsto -e_2\mapsto e_1$, as in the solution of Exercise 12.1. Note the order: the rotation applied *first* is the factor on the *right*, exactly as for matrices.

### P12.2

In [ ]:
a, b, c, d = sympy.symbols('a b c d', real=True)
q = Q(a, b, c, d)
cols = [rot(q, e) for e in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]]
R = sympy.Matrix(3, 3, lambda r, s: sympy.expand(cols[s][r]))
sympy.pprint(R)
expected = sympy.Matrix([[a**2+b**2-c**2-d**2, 2*(b*c-a*d), 2*(b*d+a*c)],
                         [2*(b*c+a*d), a**2-b**2+c**2-d**2, 2*(c*d-a*b)],
                         [2*(b*d-a*c), 2*(c*d+a*b), a**2-b**2-c**2+d**2]])
print('matches Exercise 12.3:', sympy.simplify(R - expected) == sympy.zeros(3, 3))
n = a**2 + b**2 + c**2 + d**2
print('R^T R = N(q)^2 I:', sympy.simplify(R.T * R - n**2 * sympy.eye(3)) == sympy.zeros(3, 3))
print('det R = N(q)^3:', sympy.factor(R.det()))
print('trace on the unit sphere:', sympy.simplify(R.trace().subs(b**2, 1 - a**2 - c**2 - d**2)))
Rm = sympy.Matrix(3, 3, lambda r, s: sympy.expand(rot(-q, [(1,0,0),(0,1,0),(0,0,1)][s])[r]))
print('q and -q give the same rotation:', Rm == R)

Without assuming $N(q)=1$ we get $R^TR=N(q)^2I$ and $\det R=N(q)^3$: for a unit quaternion $R$ is a rotation. The entries are quadratic in $q$, which is why $q$ and $-q$ give the same matrix: the two-to-one map of Theorem 12.4, visible in the formula itself.

### P12.3

In [ ]:
rng = np.random.default_rng(1)
steps = 100_000
axes = rng.normal(size=(steps, 3)); axes /= np.linalg.norm(axes, axis=1)[:, None]
angles = rng.normal(scale=0.05, size=steps)

def quat_of(axis, ang, dt):
    h = ang / 2
    return np.array([math.cos(h), *(math.sin(h) * axis)], dtype=dt)

def qmul(p, q):
    a, b, c, d = p; e, f, g, h = q
    return np.array([a*e - b*f - c*g - d*h, a*f + b*e + c*h - d*g,
                     a*g - b*h + c*e + d*f, a*h + b*g - c*f + d*e], dtype=p.dtype)

def mat_of(q):
    a, b, c, d = q
    return np.array([[a*a+b*b-c*c-d*d, 2*(b*c-a*d), 2*(b*d+a*c)],
                     [2*(b*c+a*d), a*a-b*b+c*c-d*d, 2*(c*d-a*b)],
                     [2*(b*d-a*c), 2*(c*d+a*b), a*a-b*b-c*c+d*d]], dtype=q.dtype)

dt = np.float32
qs = [quat_of(axes[t], angles[t], dt) for t in range(steps)]
Ms = [mat_of(x) for x in qs]
M, qq = np.eye(3, dtype=dt), np.array([1, 0, 0, 0], dtype=dt)
qq_fixed = qq.copy()
drift_M, drift_q = [], []
for t in range(steps):
    M = Ms[t] @ M
    qq = qmul(qs[t], qq)
    qq_fixed = qmul(qs[t], qq_fixed); qq_fixed /= np.linalg.norm(qq_fixed)
    if t % 1000 == 999:
        drift_M.append(np.linalg.norm(M.T @ M - np.eye(3)))
        drift_q.append(abs(np.linalg.norm(qq) - 1))
print('after', steps, 'steps (float32, no correction):')
print('  matrix  ||R^T R - I|| =', drift_M[-1])
print('  quaternion |N(q)^(1/2) - 1| =', drift_q[-1])
# the renormalised quaternion and the uncorrected one describe the same rotation up to scale
print('  angle between corrected and uncorrected quaternion directions:',
      math.degrees(math.acos(min(1, abs(float(qq @ qq_fixed) / np.linalg.norm(qq))))), 'degrees')
U, S, Vt = np.linalg.svd(M)
print('  singular values of the matrix product:', S, ' -> nearest rotation U Vt')
fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(1, len(drift_M) + 1) * 1000
ax.semilogy(x, drift_M, label='matrices: $\\|R^TR-I\\|$')
ax.semilogy(x, drift_q, label='quaternions: $|\\,|q|-1|$')
ax.set_xlabel('number of compositions'); ax.legend(); plt.show()
print('multiplications per composition: matrix 27, quaternion 16 (+ 4 mult, 1 sqrt, 4 div to renormalise)')

Both representations drift in single precision, roughly like a random walk in the rounding errors. The difference is in the *cure*: a quaternion that has left the unit sphere still describes a rotation, $v\mapsto qv\bar q/N(q)$, and dividing by its length puts it back at the cost of a square root; a matrix that has left $SO(3)$ is not a rotation at all (it shears), and the nearest rotation requires a Gram–Schmidt step or an SVD. Composition itself costs $16$ multiplications instead of $27$. These two facts, together with slerp (P12.4), are why attitude controllers, robots and game engines store orientations as quaternions (Shoemake, 1985).

### P12.4

In [ ]:
def slerp(q0, q1, t):
    dot = float(np.dot(q0, q1))
    if dot < 0:                       # q1 and -q1 are the same rotation: take the short way
        q1, dot = -q1, -dot
    om = math.acos(min(1.0, dot))
    if om < 1e-12: return q0
    return (math.sin((1 - t) * om) * q0 + math.sin(t * om) * q1) / math.sin(om)

def euler_to_mat(yaw, pitch, roll):
    cy, sy, cp, sp, cr, sr = map(lambda f: f, (math.cos(yaw), math.sin(yaw), math.cos(pitch),
                                             math.sin(pitch), math.cos(roll), math.sin(roll)))
    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])
    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])
    return Rz @ Ry @ Rx

def rot_angle(R):
    return math.acos(max(-1, min(1, (np.trace(R) - 1) / 2)))

def nearest_rotation(A):
    U, _, Vt = np.linalg.svd(A); return U @ Vt

e0, e1 = np.array([0.1, 0.2, 0.3]), np.array([2.6, -1.2, 2.9])
R0, R1 = euler_to_mat(*e0), euler_to_mat(*e1)
from_mat = lambda R: np.array(sympy.Quaternion.from_rotation_matrix(sympy.Matrix(R)).to_Matrix().T,
                              dtype=float).ravel()
q0, q1 = from_mat(R0), from_mat(R1)
print('check q0 -> R0:', np.allclose(mat_of(q0 / np.linalg.norm(q0)), R0))
ts = np.linspace(0, 1, 201)
paths = {
    'slerp': [mat_of(slerp(q0, q1, t)) for t in ts],
    'Euler angles': [euler_to_mat(*((1 - t) * e0 + t * e1)) for t in ts],
    'matrices (lerp, then nearest rotation)': [nearest_rotation((1 - t) * R0 + t * R1) for t in ts],
}
fig, ax = plt.subplots(figsize=(7, 3))
for name, P in paths.items():
    speed = [rot_angle(P[n + 1] @ P[n].T) / (ts[1] - ts[0]) for n in range(len(P) - 1)]
    total = sum(rot_angle(P[n + 1] @ P[n].T) for n in range(len(P) - 1))
    print(f'{name:40s} total angle travelled {math.degrees(total):7.2f} degrees')
    ax.plot(ts[:-1], np.degrees(speed), label=name)
print('angle of R1 R0^T (the shortest path):', round(math.degrees(rot_angle(R1 @ R0.T)), 2))
ax.set_xlabel('t'); ax.set_ylabel('angular speed (deg per unit t)'); ax.legend(fontsize=8); plt.show()

# gimbal lock: rank of the derivative of (yaw, pitch, roll) -> rotation
def jac_rank(e, h=1e-6):
    cols = []
    for m in range(3):
        de = np.zeros(3); de[m] = h
        cols.append(((euler_to_mat(*(e + de)) - euler_to_mat(*(e - de))) / (2 * h)).ravel())
    return np.linalg.matrix_rank(np.array(cols).T, tol=1e-6)
print('rank at a generic point:', jac_rank(np.array([0.3, 0.4, 0.5])))
print('rank at pitch = 90 degrees:', jac_rank(np.array([0.3, math.pi / 2, 0.5])))
print('yaw and roll only matter through yaw - roll at pitch 90:',
      np.allclose(euler_to_mat(0.3, math.pi / 2, 0.5), euler_to_mat(0.5, math.pi / 2, 0.7)))

Slerp follows the great circle of $S^3$ from $q_0$ to $q_1$ at constant speed and travels exactly the angle of $R_1R_0^{-1}$: it is the geodesic. Interpolating Euler angles goes the long way round at varying speed, and interpolating matrices passes through non-rotations: projected back onto $SO(3)$ it follows the same great circle as slerp, but its speed varies by a factor of about $7$ along the way (slow at the ends, fast in the middle). At pitch $90^\circ$ the Euler angle map has rank $2$: yaw and roll turn about the same axis and only their difference matters, so one degree of freedom is lost. This is *gimbal lock*, which famously worried the Apollo guidance engineers; it is not a defect of $SO(3)$ but of the chart, and no choice of three parameters avoids such singularities everywhere (Stuelpnagel, 1964). Unit quaternions cover $SO(3)$ smoothly twice and have no such singularity.

### P12.5

In [ ]:
def close(gens, tol=1e-9):
    """Closure of a finite set of float unit quaternions (as numpy arrays) under products."""
    key = lambda x: tuple(np.round(x, 7))
    elems = {key(np.array([1.0, 0, 0, 0])): np.array([1.0, 0, 0, 0])}
    frontier = list(elems.values())
    while frontier:
        new = []
        for x in frontier:
            for g in gens:
                y = qmul(g, x)
                if key(y) not in elems:
                    elems[key(y)] = y; new.append(y)
        frontier = new
        if len(elems) > 10_000: raise ValueError('not finite?')
    return list(elems.values())

def order(x):
    y, n = x.copy(), 1
    while not np.allclose(y, [1, 0, 0, 0], atol=1e-7):
        y, n = qmul(y, x), n + 1
    return n

phi = (1 + 5 ** 0.5) / 2
w = np.array([1, 1, 1, 1]) / 2
groups = {
    '2T': [w, np.array([0, 1.0, 0, 0])],
    '2O': [np.array([1, 1, 0, 0]) / 2 ** 0.5, w],
    '2I': [w, np.array([phi, 1 / phi, 1, 0]) / 2],
}
for name, gens in groups.items():
    G = close(gens)
    orders = Counter(order(x) for x in G)
    image = {tuple(np.round(mat_of(x), 7).ravel()) for x in G}
    img_orders = Counter()
    for x in G:
        R = mat_of(x); k, P = 1, R.copy()
        while not np.allclose(P, np.eye(3), atol=1e-7):
            P, k = P @ R, k + 1
        img_orders[k] += 1
    print(f'{name}: order {len(G)}, element orders {dict(sorted(orders.items()))}')
    print(f'    image in SO(3): order {len(image)}, element orders',
          {k: v // 2 for k, v in sorted(img_orders.items())})

The three groups have orders $24$, $48$ and $120$, with images of orders $12$, $24$ and $60$: the rotation groups $A_4$, $\mathfrak S_4$ and $A_5$ of Theorem 12.5. Their element orders tell them apart: the image of $2T$ has $3$ half-turns and $8$ rotations of order $3$ (Exercise 12.5), the image of $2I$ has $15$ half-turns, $20$ rotations of order $3$ and $24$ of order $5$, the class sizes of $A_5$. In each binary group, $-1$ is the *only* element of order $2$ (Exercise 12.6): a half-turn lifts to two elements of order $4$. The $24$ elements of $2T$ are the units of the Hurwitz quaternions, the $24$ vertices of the regular $24$-cell; the $120$ elements of $2I$ are the vertices of the $600$-cell.

### P12.6

In [ ]:
H = Fraction(1, 2)

def hurwitz_round(x):
    """Nearest Hurwitz quaternion to a quaternion with Fraction coordinates."""
    c1 = Q(*[Fraction(round(t)) for t in x.tuple()])
    c2 = Q(*[Fraction(math.floor(t)) + H for t in x.tuple()])
    return min(c1, c2, key=lambda r: (x - r).norm())

def left_gcd(x, y):
    """A generator of the left ideal Hx + Hy: Euclid with x = q y + r, N(r) < N(y)."""
    while y.norm() != 0:
        q = hurwitz_round(x * y.inv())
        x, y = y, x - q * y
    return x

UNITS = [Q(*[Fraction(s) for s in v]) for v in
         [(1,0,0,0),(-1,0,0,0),(0,1,0,0),(0,-1,0,0),(0,0,1,0),(0,0,-1,0),(0,0,0,1),(0,0,0,-1)]] + \
        [Q(*[H * s for s in signs]) for signs in itertools.product((1, -1), repeat=4)]

def four_squares(p):
    # find x, y with 1 + x^2 + y^2 = 0 mod p: try random x until -1-x^2 is a square
    if p == 2: return (1, 1, 0, 0)
    while True:
        x = random.randrange(p)
        r = (-1 - x * x) % p
        if r == 0: y = 0; break
        if pow(r, (p - 1) // 2, p) == 1:
            y = sympy.sqrt_mod(r, p); break
    g = left_gcd(Q(Fraction(p)), Q(Fraction(1), Fraction(x), Fraction(y)))
    assert g.norm() == p
    for u in UNITS:
        h = g * u
        if all(t.denominator == 1 for t in h.tuple()):
            return tuple(abs(int(t)) for t in h.tuple())

random.seed(0)
for p in [2, 3, 7, 101, 10007, 2**61 - 1]:
    s = four_squares(p)
    print(p, '=', ' + '.join(f'{t}^2' for t in s), '  check:', sum(t * t for t in s) == p)
p = sympy.nextprime(10**30)
t0 = time.perf_counter(); s = four_squares(p); dt_ = time.perf_counter() - t0
print(f'p = {p}: {s}  check {sum(t*t for t in s) == p}  in {dt_*1000:.1f} ms')

def r4(n):
    return sum(1 for v in itertools.product(range(-int(n**0.5), int(n**0.5) + 1), repeat=4)
               if sum(t * t for t in v) == n)
jacobi = lambda n: 8 * sum(d for d in sympy.divisors(n) if d % 4)
print('n, r4(n), Jacobi:', [(n, r4(n), jacobi(n)) for n in range(1, 13)])

The algorithm is the quaternion analogue of the Gaussian-integer method for two squares: one modular square root (random search plus `sqrt_mod`), then a Euclidean algorithm whose remainders shrink in norm by a factor of at least $2$ each step, because every quaternion lies within norm $\frac12$ of a Hurwitz quaternion (the proof of Exercise 12.7 only needs a bound $<1$). So $O(\log p)$ steps suffice, and a $100$-bit prime is written as a sum of four squares in milliseconds, whereas Lagrange's descent from Volume 1 would need about $p$ steps in the worst case. The final multiplication by a unit is needed because a generator may have half-integer coordinates: among $g$ and its $24$ associates $gu$, at least one has integer coordinates.

The last line checks Jacobi's four-square theorem $r_4(n)=8\sum_{4\nmid d\mid n}d$ for $n\le12$. It is a culture result here; its classical proof uses modular forms, and Hurwitz gave one by counting quaternions of norm $n$ with the unique factorisation of Exercise 12.7.